## 🚀 Convert Pascal VOC Annotations to YOLO Format & Split Dataset

This cell performs the following preprocessing steps:

1. **Parse Pascal VOC XML**  
   - Reads each `<object>` annotation from `train_labels/` and `valid_labels/`.  
   - Extracts filename, image size, class name, and bounding box coordinates.

2. **Gather & Validate**  
   - Matches each XML to its corresponding image (`.jpg`, `.png`, etc.).  
   - Warns if any annotation has no matching image.

3. **Train/Val/Test Split**  
   - Combines all records, then splits into **70% train**, **20% val**, **10% test** using `sklearn.model_selection.train_test_split`.

4. **VOC → YOLO Conversion**  
   - Converts `(xmin, ymin, xmax, ymax)` → normalized `(x_center, y_center, width, height)`.  
   - Writes one `.txt` label file per image in the corresponding `split/labels/` folder.

5. **Image Copying**  
   - Copies each image into `split/images/`, preserving original filenames.

### After running this cell, you will have:
      resnet_dataset_top100/
         ├── train/
         │   ├── Images/
         │   ├── Labels/
         └── test/
            ├── Images/
            ├── Labels/


In [ ]:
import os
import glob
import shutil
import xml.etree.ElementTree as ET
import pandas as pd
from sklearn.model_selection import train_test_split
from PIL import Image

INPUT_DIR = "/kaggle/input/cub-200-bird-species-xml-detection-dataset"
BASE_XML_DIR = os.path.join(INPUT_DIR, "cub_200_2011_xml")

TRAIN_IMAGES_DIR = os.path.join(BASE_XML_DIR, "train_images")
VALID_IMAGES_DIR = os.path.join(BASE_XML_DIR, "valid_images")
TRAIN_LABELS_DIR = os.path.join(BASE_XML_DIR, "train_labels")
VALID_LABELS_DIR = os.path.join(BASE_XML_DIR, "valid_labels")

OUTPUT_DIR = "/kaggle/working"

for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(OUTPUT_DIR, split, "images"), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, split, "labels"), exist_ok=True)

SPLIT_RATIOS = (0.7, 0.2, 0.1)  # 70% / 20% / 10%


In [ ]:
def parse_voc_xml(xml_file):
    """
    Parse a single Pascal VOC XML annotation file.
    Returns a dict with: filename, width, height, class_name, (x_min, y_min, x_max, y_max).
    Assumes exactly one <object> per XML based on your example.
    """
    tree = ET.parse(xml_file)
    root = tree.getroot()
    
    filename = root.find('filename').text 
    size = root.find('size')
    width = int(size.find('width').text)
    height = int(size.find('height').text)
    
    obj = root.find('object')
    class_name = obj.find('name').text     
    bndbox = obj.find('bndbox')
    x_min = float(bndbox.find('xmin').text)
    y_min = float(bndbox.find('ymin').text)
    x_max = float(bndbox.find('xmax').text)
    y_max = float(bndbox.find('ymax').text)
    
    return {
        "filename": filename,
        "width": width,
        "height": height,
        "class_name": class_name,
        "x_min": x_min,
        "y_min": y_min,
        "x_max": x_max,
        "y_max": y_max
    }

def gather_annotations(labels_dir, images_dir):
    """
    Go through all XML files in labels_dir and match them to images in images_dir.
    Return a list of annotation dicts.
    """
    annotation_list = []
    
    # Each XML in this directory
    xml_files = glob.glob(os.path.join(labels_dir, "*.xml"))
    for xml_file in xml_files:
        record = parse_voc_xml(xml_file)
        
        possible_exts = [".jpg", ".jpeg", ".png"]
        image_path = None
        for ext in possible_exts:
            candidate = os.path.join(images_dir, record["filename"] + ext)
            if os.path.exists(candidate):
                image_path = candidate
                break
        
        if image_path is not None:
            record["image_path"] = image_path
            annotation_list.append(record)
        else:
            print(f"Warning: No matching image found for {record['filename']} in {images_dir}")
    
    return annotation_list

# Gather train + valid data
train_records = gather_annotations(TRAIN_LABELS_DIR, TRAIN_IMAGES_DIR)
valid_records = gather_annotations(VALID_LABELS_DIR, VALID_IMAGES_DIR)

all_data = train_records + valid_records
print(f"Total annotations found: {len(all_data)}")


Total annotations found: 11787


In [3]:
# Extract all unique class names
all_class_names = sorted(set([d["class_name"] for d in all_data]))

# Create a dict mapping class_name -> class_id
class_to_id = {cls_name: idx for idx, cls_name in enumerate(all_class_names)}

print(f"Found {len(all_class_names)} unique classes.")


Found 200 unique classes.


In [ ]:
df = pd.DataFrame(all_data)

train_ratio, val_ratio, test_ratio = SPLIT_RATIOS

# First, split off train
train_df, temp_df = train_test_split(
    df, test_size=(1 - train_ratio), random_state=42, shuffle=True
)

# Next, split temp_df into val and test
remaining = val_ratio + test_ratio
val_size = val_ratio / remaining  # fraction of the remaining
val_df, test_df = train_test_split(
    temp_df, test_size=(test_ratio / remaining), random_state=42, shuffle=True
)

print(f"Train set: {len(train_df)}")
print(f"Val set:   {len(val_df)}")
print(f"Test set:  {len(test_df)}")


Train set: 8250
Val set:   2358
Test set:  1179


In [5]:
def voc_to_yolo(x_min, y_min, x_max, y_max, img_w, img_h):
    """
    Convert VOC bounding box (x_min, y_min, x_max, y_max) to
    YOLO format (x_center, y_center, width, height) normalized to [0,1].
    """
    x_center = (x_min + x_max) / 2.0 / img_w
    y_center = (y_min + y_max) / 2.0 / img_h
    w = (x_max - x_min) / img_w
    h = (y_max - y_min) / img_h
    return x_center, y_center, w, h

def process_split(df_split, split_name):
    """
    For each record in df_split:
      1) Convert bounding box to YOLO format
      2) Copy the image to /kaggle/working/<split>/images
      3) Create a .txt label file in /kaggle/working/<split>/labels
    """
    split_img_dir = os.path.join(OUTPUT_DIR, split_name, "images")
    split_label_dir = os.path.join(OUTPUT_DIR, split_name, "labels")
    
    for _, row in df_split.iterrows():
        img_path = row["image_path"]
        class_name = row["class_name"]
        class_id = class_to_id[class_name]
        
        # Convert bounding box
        x_center, y_center, w, h = voc_to_yolo(
            row["x_min"], row["y_min"], row["x_max"], row["y_max"],
            row["width"], row["height"]
        )
        
        # Construct the YOLO annotation line
        yolo_line = f"{class_id} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}\n"
        
        # Copy image
        base_name = os.path.basename(img_path)  # e.g. Acadian_Flycatcher_0003_29094.jpg
        dst_img_path = os.path.join(split_img_dir, base_name)
        shutil.copy(img_path, dst_img_path)
        
        # Write label file
        label_file = os.path.splitext(base_name)[0] + ".txt"
        dst_label_path = os.path.join(split_label_dir, label_file)
        with open(dst_label_path, "w") as f:
            f.write(yolo_line)

# Process each split
process_split(train_df, "train")
process_split(val_df,   "val")
process_split(test_df,  "test")

print("Dataset successfully converted to YOLO format!")


Dataset successfully converted to YOLO format!


## 📝 Generate `dataset.yaml` for YOLO Training

This cell writes out a `dataset.yaml` file that points YOLOv5/YOLOv8 at your new data splits:

- **`path`**: Base working directory (`/kaggle/working`)
- **`train`, `val`, `test`**: Relative paths to image folders
- **`nc`**: Number of classes (200)
- **`names`**: List of class names in index order

Place this Markdown **above** the snippet that writes `dataset.yaml` so users know:

```text
/kaggle/working/dataset.yaml


In [6]:
import os

# Define the YAML content as a multiline string
yaml_content = """path: /kaggle/working
train: train/images  # Path to training images
val: val/images      # Path to validation images
test: test/images    # Path to test images (optional)

# Number of classes
nc: 200

# Class names (map index to bird species)
names:
  - Black_footed_Albatross
  - Laysan_Albatross
  - Sooty_Albatross
  - Groove_billed_Ani
  - Crested_Auklet
  - Least_Auklet
  - Parakeet_Auklet
  - Rhinoceros_Auklet
  - Brewer_Blackbird
  - Red_winged_Blackbird
  - Rusty_Blackbird
  - Yellow_headed_Blackbird
  - Bobolink
  - Indigo_Bunting
  - Lazuli_Bunting
  - Painted_Bunting
  - Cardinal
  - Spotted_Catbird
  - Gray_Catbird
  - Yellow_breasted_Chat
  - Eastern_Towhee
  - Chuck_will_Widow
  - Brandt_Cormorant
  - Red_faced_Cormorant
  - Pelagic_Cormorant
  - Bronzed_Cowbird
  - Shiny_Cowbird
  - Brown_Creeper
  - American_Crow
  - Fish_Crow
  - Black_billed_Cuckoo
  - Mangrove_Cuckoo
  - Yellow_billed_Cuckoo
  - Gray_crowned_Rosy_Finch
  - Purple_Finch
  - Northern_Flicker
  - Acadian_Flycatcher
  - Great_Crested_Flycatcher
  - Least_Flycatcher
  - Olive_sided_Flycatcher
  - Scissor_tailed_Flycatcher
  - Vermilion_Flycatcher
  - Yellow_bellied_Flycatcher
  - Frigatebird
  - Northern_Fulmar
  - Gadwall
  - American_Goldfinch
  - European_Goldfinch
  - Boat_tailed_Grackle
  - Eared_Grebe
  - Horned_Grebe
  - Pied_billed_Grebe
  - Western_Grebe
  - Blue_Grosbeak
  - Evening_Grosbeak
  - Pine_Grosbeak
  - Rose_breasted_Grosbeak
  - Pigeon_Guillemot
  - California_Gull
  - Glaucous_winged_Gull
  - Heermann_Gull
  - Herring_Gull
  - Ivory_Gull
  - Ring_billed_Gull
  - Slaty_backed_Gull
  - Western_Gull
  - Anna_Hummingbird
  - Ruby_throated_Hummingbird
  - Rufous_Hummingbird
  - Green_Violetear
  - Long_tailed_Jaeger
  - Pomarine_Jaeger
  - Blue_Jay
  - Florida_Jay
  - Green_Jay
  - Dark_eyed_Junco
  - Tropical_Kingbird
  - Gray_Kingbird
  - Belted_Kingfisher
  - Green_Kingfisher
  - Pied_Kingfisher
  - Ringed_Kingfisher
  - White_breasted_Kingfisher
  - Red_legged_Kittiwake
  - Horned_Lark
  - Pacific_Loon
  - Mallard
  - Western_Meadowlark
  - Hooded_Merganser
  - Red_breasted_Merganser
  - Mockingbird
  - Nighthawk
  - Clark_Nutcracker
  - White_breasted_Nuthatch
  - Baltimore_Oriole
  - Hooded_Oriole
  - Orchard_Oriole
  - Scott_Oriole
  - Ovenbird
  - Brown_Pelican
  - White_Pelican
  - Western_Wood_Pewee
  - Sayornis
  - American_Pipit
  - Whip_poor_Will
  - Horned_Puffin
  - Common_Raven
  - White_necked_Raven
  - American_Redstart
  - Geococcyx
  - Loggerhead_Shrike
  - Great_Grey_Shrike
  - Baird_Sparrow
  - Black_throated_Sparrow
  - Brewer_Sparrow
  - Chipping_Sparrow
  - Clay_colored_Sparrow
  - House_Sparrow
  - Field_Sparrow
  - Fox_Sparrow
  - Grasshopper_Sparrow
  - Harris_Sparrow
  - Henslow_Sparrow
  - Le_Conte_Sparrow
  - Lincoln_Sparrow
  - Nelson_Sharp_tailed_Sparrow
  - Savannah_Sparrow
  - Seaside_Sparrow
  - Song_Sparrow
  - Tree_Sparrow
  - Vesper_Sparrow
  - White_crowned_Sparrow
  - White_throated_Sparrow
  - Cape_Glossy_Starling
  - Bank_Swallow
  - Barn_Swallow
  - Cliff_Swallow
  - Tree_Swallow
  - Scarlet_Tanager
  - Summer_Tanager
  - Artic_Tern
  - Black_Tern
  - Caspian_Tern
  - Common_Tern
  - Elegant_Tern
  - Forsters_Tern
  - Least_Tern
  - Green_tailed_Towhee
  - Brown_Thrasher
  - Sage_Thrasher
  - Black_capped_Vireo
  - Blue_headed_Vireo
  - Philadelphia_Vireo
  - Red_eyed_Vireo
  - Warbling_Vireo
  - White_eyed_Vireo
  - Yellow_throated_Vireo
  - Bay_breasted_Warbler
  - Black_and_white_Warbler
  - Black_throated_Blue_Warbler
  - Blue_winged_Warbler
  - Canada_Warbler
  - Cape_May_Warbler
  - Cerulean_Warbler
  - Chestnut_sided_Warbler
  - Golden_winged_Warbler
  - Hooded_Warbler
  - Kentucky_Warbler
  - Magnolia_Warbler
  - Mourning_Warbler
  - Myrtle_Warbler
  - Nashville_Warbler
  - Orange_crowned_Warbler
  - Palm_Warbler
  - Pine_Warbler
  - Prairie_Warbler
  - Prothonotary_Warbler
  - Swainson_Warbler
  - Tennessee_Warbler
  - Wilson_Warbler
  - Worm_eating_Warbler
  - Yellow_Warbler
  - Northern_Waterthrush
  - Louisiana_Waterthrush
  - Bohemian_Waxwing
  - Cedar_Waxwing
  - American_Three_toed_Woodpecker
  - Pileated_Woodpecker
  - Red_bellied_Woodpecker
  - Red_cockaded_Woodpecker
  - Red_headed_Woodpecker
  - Downy_Woodpecker
  - Bewick_Wren
  - Cactus_Wren
  - Carolina_Wren
  - House_Wren
  - Marsh_Wren
  - Rock_Wren
  - Winter_Wren
  - Common_Yellowthroat
"""

# Path to save the file
yaml_file_path = "/kaggle/working/dataset.yaml"

# Write the file
with open(yaml_file_path, "w") as f:
    f.write(yaml_content)

print(f"YAML file successfully written to: {yaml_file_path}")


YAML file successfully written to: /kaggle/working/dataset.yaml


# YOLO Model Training Documentation

In this guide, we outline the steps and parameters used to train a YOLO model for object detection. The model is fine-tuned using a pretrained weight file and configured for training on a custom dataset.

## Overview

We are using the YOLO model with the following training parameters:

- **Custom Dataset:** Training is done on a dataset specified by the `dataset.yaml` configuration file.


In [7]:
!pip install ultralytics --upgrade

from ultralytics import YOLO


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 949.8/949.8 kB 23.9 MB/s eta 0:00:0000:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [8]:
model = YOLO('/kaggle/input/epoch25.pt/pytorch/default/1/epoch25.pt')

In [9]:
model.train(
    data="/kaggle/working/dataset.yaml",  # Path to your YAML configuration
    epochs=20,                            # Set the number of training epochs
    imgsz=500,                           # Set image size to 500 (max dimension in your dataset)                           # Adjust batch size as needed (e.g., for multi-GPU use)
    device="cuda",                        # Use both GPUs
    save_period=5,                       # Save model checkpoint every 5 epochs
    name="yolo11l_cub200"                # Name of the training run/output folder
)

Ultralytics 8.3.94 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=/kaggle/input/epoch25.pt/pytorch/default/1/epoch25.pt, data=/kaggle/working/dataset.yaml, epochs=20, time=None, patience=100, batch=16, imgsz=500, save=True, save_period=5, cache=False, device=cuda, workers=8, project=None, name=yolo11l_cub200, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_cro

100%|██████████| 755k/755k [00:00<00:00, 23.9MB/s]



                   from  n    params  module                                       arguments                     
  0                  -1  1      1856  ultralytics.nn.modules.conv.Conv             [3, 64, 3, 2]                 
  1                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  2                  -1  2    173824  ultralytics.nn.modules.block.C3k2            [128, 256, 2, True, 0.25]     
  3                  -1  1    590336  ultralytics.nn.modules.conv.Conv             [256, 256, 3, 2]              
  4                  -1  2    691712  ultralytics.nn.modules.block.C3k2            [256, 512, 2, True, 0.25]     
  5                  -1  1   2360320  ultralytics.nn.modules.conv.Conv             [512, 512, 3, 2]              
  6                  -1  2   2234368  ultralytics.nn.modules.block.C3k2            [512, 512, 2, True]           
  7                  -1  1   2360320  ultralytics.nn.modules.conv.Conv             [512

100%|██████████| 5.35M/5.35M [00:00<00:00, 112MB/s]


AMP: checks passed ✅
WARNING ⚠️ imgsz=[500] must be multiple of max stride 32, updating to [512]


...

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 74/74 [00:24<00:00,  3.02it/s]


                   all       2358       2358      0.838      0.812      0.864       0.79
Black_footed_Albatross          9          9      0.638      0.556      0.678      0.642
      Laysan_Albatross         13         13      0.355      0.462      0.482      0.471
       Sooty_Albatross         10         10      0.895      0.853      0.912      0.853
     Groove_billed_Ani          8          8          1       0.97      0.995      0.906
        Crested_Auklet         12         12      0.864          1      0.995      0.846
          Least_Auklet         11         11          1      0.888      0.995      0.748
       Parakeet_Auklet         15         15      0.894      0.733      0.908      0.855
     Rhinoceros_Auklet         17         17      0.707      0.709       0.76      0.741
      Brewer_Blackbird         18         18      0.886      0.868      0.935      0.785
  Red_winged_Blackbird         10         10          1      0.851      0.928      0.888
       Rusty_Blackbir

/usr/local/lib/python3.10/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.10/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


Speed: 0.1ms preprocess, 7.2ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to runs/detect/yolo11l_cub200


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,  53,  54,  55,  56,  57,  58,  59,  60,  61,
        62,  63,  64,  65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,  78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,  91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123,
       124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 1

# YOLOv8 Training Summary

The model was trained for 20 epochs on the **CUB-200 dataset** with the following settings:
- **Image size**: 512x512
- **Optimizer**: AdamW
- **Batch size**: 10
- **Epochs**: 20

## Key Metrics (mAP50, mAP50-95)
| Epoch | Box Loss | Class Loss | DFL Loss | mAP50 | mAP50-95 |
|-------|----------|------------|----------|-------|----------|
| 1     | 0.6024   | 0.9808     | 1.098    | 0.853 | 0.774    |
| 2     | 0.6003   | 0.941      | 1.103    | 0.843 | 0.758    |
| 3     | 0.609    | 0.9793     | 1.105    | 0.833 | 0.743    |
| 4     | 0.6135   | 0.9985     | 1.108    | 0.836 | 0.748    |
| 5     | 0.604    | 0.9516     | 1.102    | 0.84  | 0.752    |
| 6     | 0.5949   | 0.9149     | 1.097    | 0.839 | 0.749    |
| 7     | 0.5915   | 0.8783     | 1.095    | 0.847 | 0.761    |
| 8     | 0.5888   | 0.8616     | 1.092    | 0.847 | 0.764    |
| 9     | 0.5656   | 0.8098     | 1.076    | 0.852 | 0.773    |
| 10    | 0.5595   | 0.7696     | 1.075    | 0.857 | 0.774    |
| 11    | 0.3795   | 0.3561     | 0.9493    | 0.844 | 0.761    |
| 12    | 0.3714   | 0.3259     | 0.9446    | 0.845 | 0.761    |
| 13    | 0.3621   | 0.3026     | 0.9333    | 0.852 | 0.773    |
| 14    | 0.3541   | 0.2877     | 0.9253    | 0.852 | 0.775    |
| 15    | 0.3515   | 0.2643     | 0.9197    | 0.859 | 0.777    |
| 16    | 0.3404   | 0.2491     | 0.9151    | 0.863 | 0.785    |
| 17    | 0.3393   | 0.2377     | 0.9116    | 0.862 | 0.786    |
| 18    | 0.3305   | 0.2244     | 0.906     | 0.864 | 0.788    |
| 19    | 0.3224   | 0.2107     | 0.9007    | 0.862 | 0.788    |
| 20    | 0.3185   | 0.2043     | 0.8949    | 0.864 | 0.790    |

## Final Model Performance:
- **Final mAP50**: 0.864
- **Final mAP50-95**: 0.790
- **Box Precision**: 0.836
- **Class Precision**: 0.813
- **Class Recall**: 0.821

---

### Notes:
- The training used **4 dataloader workers** for data loading.
- Image augmentation (using `albumentations`) was applied.
- The optimizer, **AdamW**, was chosen for automatic learning rate adjustment.
- **Training time**: Approximately **1.5 hours** for 20 epochs.

### Recommendations:
- The model shows good improvement in both precision and recall over time, with **mAP50** improving steadily.
- The final model is suitable for deployment or further fine-tuning depending on specific needs.
